# TF-PWA to amplitude-serialization converter

This notebook converts a TF-PWA analysis with the same structural conventions as this repository into the RUB-EP1 amplitude-serialization JSON format.

The notebook is intentionally split into small functions and one wrapper call. The first section optionally runs the original TF-PWA model to produce complex reference amplitudes. The conversion itself reads the TF-PWA configuration and parameter files, extracts the active decay chains, and builds a schema-shaped JSON document.

Important convention used here:

- `0 = Bp`
- `1 = D0`
- `2 = pi`
- `3 = D`
- `4 = K`

With that convention the validated topologies are:

- D*D chains: `[[[1, 2], 3], 4]`
- DK / X(2900) chains: `[[1, 2], [3, 4]]`

## Scope and limitations

This notebook currently targets TF-PWA analyses with the same cascade syntax used here: `B -> D*D K` and `B -> DK D*`, with `D* -> D0 pi`. It is written so path handling, parameter loading, chain extraction, and JSON assembly can be reused for analogous analyses, but the topology resolver and TF-PWA parameter-key resolver are intentionally explicit and should be reviewed for a new decay.

Physics fidelity is marked in the output warnings. Plain `BWR` chains are emitted as schema built-ins where possible. TF-PWA-specific structures such as `BWR_LS` shared denominators, below-threshold `q0` handling, and nonresonant `New` terms are emitted as `custom` functions with metadata-style expressions until an evaluator implements these conventions. The generated JSON is therefore a structured model description, not yet a guaranteed independent evaluator of every amplitude component.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import yaml

## User-facing configuration

Change only these paths/options for normal use. The reference amplitude computation is optional; the converter can build the JSON without sampling events.

In [ ]:
@dataclass(frozen=True)
class ConverterPaths:
    repo_root: Path
    analysis_dir: Path
    config_path: Path
    params_path: Path
    output_path: Path


def default_paths() -> ConverterPaths:
    repo_root = Path.cwd()
    if repo_root.name == "Analysis":
        repo_root = repo_root.parent
    if not (repo_root / "Analysis" / "config_a.yml").exists():
        repo_root = Path(r"C:\Users\gamma\Documents\Playground\B2DxDK.jl_fresh")
    analysis_dir = repo_root / "Analysis"
    return ConverterPaths(
        repo_root=repo_root,
        analysis_dir=analysis_dir,
        config_path=analysis_dir / "config_a.yml",
        params_path=analysis_dir / "final_params_full.json",
        output_path=analysis_dir / "tfpwa_amplitude_serialization_generated.json",
    )


PATHS = default_paths()
FINAL_STATE_ORDER = ["D0", "pi", "D", "K"]
REFERENCE_TOPOLOGY = [[[1, 2], 3], 4]
COMPUTE_REFERENCE_AMPLITUDES = True
REFERENCE_EVENT_COUNT = 3

PATHS

## Minimal TF-PWA reference amplitudes

This section executes the original TF-PWA model in the smallest practical way: it samples a few phase-space points, computes TF-PWA angles, loads the parameter values, and calls `get_amp` for the coherent model. These values are stored in the generated JSON under `misc.reference_tfpwa` so later implementations can reproduce the same check.

In [ ]:
def ensure_tfpwa_on_path(paths: ConverterPaths) -> None:
    tfpwa_src = paths.repo_root / "tf-pwa"
    for candidate in [tfpwa_src, paths.analysis_dir]:
        candidate_str = str(candidate)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)


def load_yaml(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_tfpwa_config_yaml(path: Path) -> dict[str, Any]:
    """Load a TF-PWA YAML file and inline the particle `$include` block.

    TF-PWA resolves includes internally. The converter needs the same particle
    metadata, otherwise resonance spins/masses can silently fall back to defaults.
    """
    config = load_yaml(path)
    particle = dict(config.get("particle", {}))
    include_name = particle.get("$include")
    if include_name:
        included = load_yaml(path.parent / include_name)
        merged_particle = dict(included)
        merged_particle.update({key: value for key, value in particle.items() if key != "$include"})
        config = dict(config)
        config["particle"] = merged_particle
    return config


def load_params(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    return data.get("value", data)


def tfpwa_nominal_final_masses(config_yml: dict[str, Any], dat_order: list[str]) -> dict[str, float]:
    finals = config_yml["particle"]["$finals"]
    return {name: float(finals[name]["mass"]) for name in dat_order}


def compute_tfpwa_reference_amplitudes(
    paths: ConverterPaths,
    n_events: int = 3,
    chain_spec: str = "all",
    seed: int = 12345,
) -> dict[str, Any]:
    """Compute a tiny deterministic TF-PWA amplitude sample for validation.

    This is deliberately independent of conversion. If TensorFlow/TF-PWA is not
    available in the active kernel, the wrapper can skip this step.
    """
    ensure_tfpwa_on_path(paths)
    cwd = Path.cwd()
    os.chdir(paths.analysis_dir)
    try:
        import tensorflow as tf
        import extra_amp  # noqa: F401, required because config_a.yml references custom models
        from tf_pwa.config_loader import ConfigLoader
        from tf_pwa.phasespace import PhaseSpaceGenerator

        config_yml = load_tfpwa_config_yaml(paths.config_path)
        params = load_params(paths.params_path)
        dat_order = list(config_yml["data"]["dat_order"])
        masses = tfpwa_nominal_final_masses(config_yml, dat_order)

        tf.random.set_seed(seed)
        generator = PhaseSpaceGenerator(
            float(config_yml["particle"]["$top"]["Bp"]["mass"]),
            [masses[name] for name in dat_order],
        )
        p4_arrays = [np.asarray(v) for v in generator.generate(n_events)]
        sampled_p4 = dict(zip(dat_order, [v.tolist() for v in p4_arrays]))

        config = ConfigLoader(str(paths.config_path.name))
        particle_map = {p.name: p for p in list(config.get_decay().outs)}
        p4_tfpwa = {
            particle_map[name]: tf.constant(sampled_p4[name], dtype=tf.float64)
            for name in dat_order
        }
        phsp_variables = config.data.cal_angle(p4_tfpwa)
        phsp_variables["c"] = np.full(n_events, config_yml.get("data", {}).get("extra_var", {}).get("c", {}).get("default", -1.0))

        amp_model = config.get_amplitude()
        decay_group = amp_model.decay_group
        config.set_params(params)
        if chain_spec == "all":
            decay_group.set_used_chains(list(range(len(decay_group.chains))))
        else:
            decay_group.set_used_chains([int(chain_spec)])
        amplitudes = decay_group.get_amp(phsp_variables).numpy().reshape(-1)

        return {
            "seed": seed,
            "chain_spec": chain_spec,
            "dat_order": dat_order,
            "p4": sampled_p4,
            "amplitudes": [
                {"real": float(np.real(value)), "imag": float(np.imag(value))}
                for value in amplitudes
            ],
        }
    finally:
        os.chdir(cwd)

In [ ]:
reference_tfpwa = None
if COMPUTE_REFERENCE_AMPLITUDES:
    try:
        reference_tfpwa = compute_tfpwa_reference_amplitudes(PATHS, n_events=REFERENCE_EVENT_COUNT)
        print("Reference TF-PWA amplitudes:")
        for idx, value in enumerate(reference_tfpwa["amplitudes"], start=1):
            print(f"  event {idx}: {value['real']:+.12e} {value['imag']:+.12e}i")
    except Exception as exc:
        print("Reference TF-PWA amplitude computation failed and will be skipped.")
        print(f"  {type(exc).__name__}: {exc}")
        reference_tfpwa = {"status": "failed", "error": f"{type(exc).__name__}: {exc}"}
else:
    print("Reference TF-PWA amplitude computation skipped.")

## Extract TF-PWA analysis information

The converter uses TF-PWA's own `DecayGroup.chains` as the primary structural source. YAML and JSON files provide particle properties, model names, nominal masses, widths, and fitted couplings.

In [ ]:
def spin_to_string(value: Any) -> str:
    if value is None:
        return "0"
    if isinstance(value, str):
        return value.replace("+", "")
    numeric = float(value)
    if abs(numeric - round(numeric)) < 1e-12:
        return str(int(round(numeric)))
    if abs(2 * numeric - round(2 * numeric)) < 1e-12:
        return f"{int(round(2 * numeric))}/2"
    return str(value)


def parameter_aliases(name: str) -> list[str]:
    """Resolve TF-PWA display/core names to parameter-file naming variants."""
    aliases = [name]
    aliases.append(name.replace("(1+)", "(1.)"))
    aliases.append(name.replace("(1-)", "(1-)"))
    aliases.append(name.replace("(0+)", "(0.)"))
    aliases.append(name.replace("(0-)", "(0-)"))
    return list(dict.fromkeys(aliases))


def first_existing_key(params: dict[str, Any], key: str) -> str | None:
    if key in params:
        return key
    for original, alias in [("(1+)", "(1.)"), ("(0+)", "(0.)")]:
        candidate = key.replace(original, alias)
        if candidate in params:
            return candidate
    return None


def particle_info(config_yml: dict[str, Any], params: dict[str, Any], name: str) -> dict[str, Any]:
    particle_cfg = config_yml["particle"]
    if name == "Bp":
        block = particle_cfg["$top"][name]
    elif name in particle_cfg.get("$finals", {}):
        block = particle_cfg["$finals"][name]
    else:
        block = {}
        for alias in parameter_aliases(name):
            if alias in particle_cfg:
                block = particle_cfg[alias]
                break
    mass = None
    for alias in parameter_aliases(name):
        mass_key = first_existing_key(params, f"{alias}_mass")
        if mass_key is not None:
            mass = params[mass_key]
            break
    if mass is None:
        mass = block.get("mass", block.get("m0"))
    return {
        "name": name,
        "mass": float(mass) if mass is not None else name + "_mass",
        "spin": spin_to_string(block.get("J", 0)),
        "model": block.get("model", "one"),
        "parity": block.get("P", block.get("Par")),
    }


def param_real(params: dict[str, Any], key: str, default: float | None = None) -> float | None:
    resolved = first_existing_key(params, key)
    if resolved is not None:
        return float(params[resolved])
    return default


def param_complex_polar(params: dict[str, Any], key: str, default: complex = 1.0 + 0.0j) -> complex:
    """TF-PWA stores complex variables as polar radius/phase by default."""
    r_key = first_existing_key(params, key + "r")
    phi_key = first_existing_key(params, key + "i")
    if r_key is None or phi_key is None:
        return default
    return float(params[r_key]) * np.exp(1j * float(params[phi_key]))


def complex_to_schema(value: complex) -> str:
    sign = "+" if value.imag >= 0 else "-"
    return f"{value.real:.16g} {sign} {abs(value.imag):.16g}i"


def load_tfpwa_decay_group(paths: ConverterPaths):
    ensure_tfpwa_on_path(paths)
    cwd = Path.cwd()
    os.chdir(paths.analysis_dir)
    try:
        import extra_amp  # noqa: F401
        from tf_pwa.config_loader import ConfigLoader

        config = ConfigLoader(str(paths.config_path.name))
        return config.get_decay()
    finally:
        os.chdir(cwd)


def extract_chain_steps(paths: ConverterPaths) -> list[dict[str, Any]]:
    decay_group = load_tfpwa_decay_group(paths)
    extracted = []
    for chain_idx, chain in enumerate(decay_group.chains):
        steps = []
        for decay in chain:
            steps.append({
                "core": str(decay.core),
                "outs": [str(out) for out in decay.outs],
                "ls_list": [tuple(map(int, ls)) for ls in decay.get_ls_list()],
                "model_name": getattr(decay, "model_name", None),
                "has_barrier_factor": bool(getattr(decay, "has_barrier_factor", False)),
                "barrier_factor_norm": bool(getattr(decay, "barrier_factor_norm", False)),
                "p_break": bool(getattr(decay, "p_break", False)),
            })
        extracted.append({"chain_index": chain_idx, "steps": steps})
    return extracted

In [ ]:
config_yml = load_tfpwa_config_yaml(PATHS.config_path)
params = load_params(PATHS.params_path)
chain_steps = extract_chain_steps(PATHS)

print(f"Loaded {len(chain_steps)} TF-PWA chains")
for chain in chain_steps[:3]:
    print(chain)

## Build amplitude-serialization JSON blocks

The functions below translate extracted TF-PWA structures into the schema. The notebook emits warnings for conventions that cannot be represented as a plain built-in schema object.

In [ ]:
def final_state_index_map(final_state_order: list[str]) -> dict[str, int]:
    return {name: idx for idx, name in enumerate(final_state_order, start=1)}


def make_state(index: int, info: dict[str, Any]) -> dict[str, Any]:
    return {"index": index, "name": info["name"], "spin": info["spin"], "mass": info["mass"]}


def build_kinematics(config_yml: dict[str, Any], params: dict[str, Any], final_state_order: list[str]) -> dict[str, Any]:
    initial = make_state(0, particle_info(config_yml, params, "Bp"))
    finals = [make_state(idx, particle_info(config_yml, params, name)) for idx, name in enumerate(final_state_order, start=1)]
    return {"initial_state": initial, "final_state": finals}


def node_for_core(core: str) -> list[Any] | None:
    mapping = {
        "Dst": [1, 2],
        "DstD": [[1, 2], 3],
        "DstK": [[1, 2], 4],
        "DK": [3, 4],
    }
    if core in mapping:
        return mapping[core]
    return None


def chain_topology_from_steps(steps: list[dict[str, Any]]) -> list[Any]:
    first = steps[0]
    root_daughters = set(first["outs"])
    if {"K"}.issubset(root_daughters):
        return [[[1, 2], 3], 4]
    if {"Dst"}.issubset(root_daughters) and any("X" in name for name in root_daughters):
        return [[1, 2], [3, 4]]
    raise ValueError(f"Cannot infer topology for root daughters {root_daughters}")


def resonance_name_from_steps(steps: list[dict[str, Any]]) -> str:
    for step in steps:
        core = step["core"]
        if core not in {"Bp", "Dst"}:
            return core
    raise ValueError("Could not identify resonance in TF-PWA chain")


def resonance_node_for_topology(topology: list[Any], resonance: str) -> list[Any]:
    if topology == [[[1, 2], 3], 4]:
        return [[1, 2], 3]
    if topology == [[1, 2], [3, 4]]:
        return [3, 4]
    raise ValueError(f"Unsupported topology for {resonance}: {topology}")


def root_node_for_topology(topology: list[Any]) -> list[Any]:
    return topology


def spin_for_particle(config_yml: dict[str, Any], params: dict[str, Any], name: str) -> str:
    return particle_info(config_yml, params, name)["spin"]


def tfpwa_parameter_name(name: str) -> str:
    return parameter_aliases(name)[1] if "(1+)" in name else name


def total_weight_key(resonance: str, topology: list[Any]) -> str:
    key_name = tfpwa_parameter_name(resonance)
    if topology == [[[1, 2], 3], 4]:
        return f"Bp->{key_name}.K{key_name}->Dst.DDst->D0.pi_total_0"
    if topology == [[1, 2], [3, 4]]:
        return f"Bp->{key_name}.Dst{key_name}->D.KDst->D0.pi_total_0"
    raise ValueError(f"Unsupported topology for total weight: {topology}")


def make_ls_vertex(node: list[Any], ls: tuple[int, int], formfactor: str = "") -> dict[str, Any]:
    return {
        "type": "ls",
        "node": node,
        "l": int(ls[0]),
        "s": spin_to_string(ls[1]),
        "formfactor": formfactor,
    }


def make_dst_vertex() -> dict[str, Any]:
    return make_ls_vertex([1, 2], (1, 0))


def safe_name_fragment(name: str) -> str:
    replacements = {
        "(": "",
        ")": "",
        "+": "p",
        "-": "m",
        ".": "p",
        "*": "st",
        " ": "_",
    }
    out = name
    for old, new in replacements.items():
        out = out.replace(old, new)
    return out.replace("/", "_")


def blatt_weisskopf_function_name(resonance: str, vertex_label: str, l: int) -> str:
    return f"BlattWeisskopf_{safe_name_fragment(resonance)}_{vertex_label}_L{int(l)}"


def blatt_weisskopf_function(name: str, l: int, radius: float = 3.0) -> dict[str, Any]:
    return {"name": name, "type": "BlattWeisskopf", "l": int(l), "radius": float(radius)}


def invariant_mass_squared_variable(topology: list[Any]) -> str:
    if topology == [[[1, 2], 3], 4]:
        return "m_DstD_sq"
    if topology == [[1, 2], [3, 4]]:
        return "m_DK_sq"
    raise ValueError(f"Unsupported topology for invariant-mass variable: {topology}")


def blatt_weisskopf_polynomial(l: int, z: float) -> float:
    """TF-PWA/HadronicLineshapes Blatt-Weisskopf denominator polynomial."""
    if l == 0:
        return 1.0
    if l == 1:
        return 1.0 + z
    if l == 2:
        return 9.0 + 3.0 * z + z * z
    raise ValueError(f"Blatt-Weisskopf convention factor is implemented up to L=2, got L={l}")


def two_body_momentum_abs(m0: float, m1: float, m2: float) -> float:
    q2 = ((m0 * m0 - (m1 + m2) ** 2) * (m0 * m0 - (m1 - m2) ** 2)) / (4.0 * m0 * m0)
    return float(np.sqrt(abs(q2)))


def cascade_blatt_weisskopf_value(l: int, d: float, m0: float, m1: float, m2: float) -> float:
    """Value of HadronicLineshapes BlattWeisskopf{L}(d)(m0^2,m1^2,m2^2)."""
    q = two_body_momentum_abs(m0, m1, m2)
    z = (q * d) ** 2
    return q**l / np.sqrt(blatt_weisskopf_polynomial(l, z))


def convention_mismatch_factor_for_vertex(l: int, d: float, m0: float, m1: float, m2: float) -> float:
    """Constant that converts package-native unnormalized vertices to TF-PWA-normalized ones."""
    value = cascade_blatt_weisskopf_value(int(l), d, m0, m1, m2)
    if value == 0.0:
        raise ZeroDivisionError(f"Cannot build convention mismatch factor for L={l} with q0=0")
    return 1.0 / value


def ad_hoc_mass(m0: float, m_min: float, m_max: float) -> float:
    """TF-PWA ad_hoc mass used to define q0 for below-threshold BWR_LS components."""
    k = (m_max - m_min) / 2.0
    return k * (1.0 + np.tanh((2.0 * m0 - (m_max + m_min)) / k / 4.0)) + m_min


DEFAULT_NOMINAL_MASSES = {
    "NR(1+)PSp": 4.35,
    "NR(1.)PSp": 4.35,
}


def mass_value(config_yml: dict[str, Any], params: dict[str, Any], name: str) -> float:
    mass = particle_info(config_yml, params, name)["mass"]
    if isinstance(mass, (int, float)):
        return float(mass)
    if isinstance(mass, str):
        for alias in parameter_aliases(name):
            key = first_existing_key(params, f"{alias}_mass")
            if key is not None:
                return float(params[key])
        for alias in parameter_aliases(name):
            if alias in DEFAULT_NOMINAL_MASSES:
                return DEFAULT_NOMINAL_MASSES[alias]
    return float(mass)


def decay_nominal_mass_for_mismatch(
    config_yml: dict[str, Any],
    params: dict[str, Any],
    resonance: str,
    m1: float,
    m2: float,
) -> float:
    m0 = mass_value(config_yml, params, resonance)
    if m0 >= m1 + m2:
        return m0
    # This mirrors TF-PWA's BWR_LS protection for below-threshold nominal q0.
    m_bp = mass_value(config_yml, params, "Bp")
    m_k = mass_value(config_yml, params, "K")
    return ad_hoc_mass(m0, m1 + m2, m_bp - m_k)


def convention_mismatch_factor(
    config_yml: dict[str, Any],
    params: dict[str, Any],
    resonance: str,
    topology: list[Any],
    root_ls: tuple[int, int],
    decay_ls: tuple[int, int],
    d: float = 3.0,
) -> float:
    """Product of nominal vertex factors needed to compare TF-PWA with CascadeDecays conventions.

    The serialized `weight` should carry these constants because CascadeDecays/HadronicLineshapes
    vertex form factors are unnormalized at q0, while TF-PWA normalizes the Blatt-Weisskopf ratio
    by the nominal breakup momentum of each vertex.
    """
    m_bp = mass_value(config_yml, params, "Bp")
    m_k = mass_value(config_yml, params, "K")
    m_d = mass_value(config_yml, params, "D")
    m_dst = mass_value(config_yml, params, "Dst")
    m_r = mass_value(config_yml, params, resonance)
    root_l = int(root_ls[0])
    decay_l = int(decay_ls[0])

    if topology == [[[1, 2], 3], 4]:
        decay_m0 = decay_nominal_mass_for_mismatch(config_yml, params, resonance, m_dst, m_d)
        return (
            convention_mismatch_factor_for_vertex(root_l, d, m_bp, m_r, m_k)
            * convention_mismatch_factor_for_vertex(decay_l, d, decay_m0, m_dst, m_d)
        )
    if topology == [[1, 2], [3, 4]]:
        return (
            convention_mismatch_factor_for_vertex(root_l, d, m_bp, m_r, m_dst)
            * convention_mismatch_factor_for_vertex(decay_l, d, m_r, m_d, m_k)
        )
    raise ValueError(f"Unsupported topology for convention mismatch factor: {topology}")


In [ ]:
def function_name_for_resonance(resonance: str, suffix: str = "") -> str:
    safe = resonance.replace(" ", "_").replace("+", "p").replace("-", "m")
    return f"lineshape_{safe}{suffix}"


def lineshape_mass_pair(config_yml: dict[str, Any], params: dict[str, Any], topology: list[Any]) -> tuple[float, float]:
    if topology == [[1, 2], [3, 4]]:
        return mass_value(config_yml, params, "D"), mass_value(config_yml, params, "K")
    return mass_value(config_yml, params, "Dst"), mass_value(config_yml, params, "D")


def lineshape_mass_names(topology: list[Any]) -> tuple[str, str]:
    if topology == [[1, 2], [3, 4]]:
        return "D", "K"
    return "Dst", "D"


def resonance_mass(config_yml: dict[str, Any], params: dict[str, Any], resonance: str) -> float:
    return mass_value(config_yml, params, resonance)


def resonance_width(config_yml: dict[str, Any], params: dict[str, Any], resonance: str) -> float:
    info = particle_info(config_yml, params, resonance)
    key = first_existing_key(params, f"{tfpwa_parameter_name(resonance)}_width")
    if key is not None:
        return float(params[key])
    block = config_yml["particle"].get(tfpwa_parameter_name(resonance), config_yml["particle"].get(resonance, {}))
    return float(block.get("width", block.get("g0", info.get("width", 0.0))))


def bwr_ls_gamma(params: dict[str, Any], resonance: str, l: int) -> float:
    theta0 = param_real(params, f"{tfpwa_parameter_name(resonance)}_theta0", 0.0)
    if int(l) == 0:
        return float(np.cos(theta0))
    if int(l) == 2:
        return float(np.sin(theta0))
    return 1.0


def bwr_ls_q0_mass(
    config_yml: dict[str, Any],
    params: dict[str, Any],
    resonance: str,
    topology: list[Any],
    ma: float,
    mb: float,
) -> float:
    m0 = resonance_mass(config_yml, params, resonance)
    if m0 >= ma + mb:
        return m0
    if topology == [[[1, 2], 3], 4]:
        return ad_hoc_mass(m0, ma + mb, mass_value(config_yml, params, "Bp") - mass_value(config_yml, params, "K"))
    return m0


def multichannel_breit_wigner_channels(
    config_yml: dict[str, Any],
    params: dict[str, Any],
    resonance: str,
    topology: list[Any],
    all_decay_ls: list[tuple[int, int]],
) -> list[dict[str, Any]]:
    m0 = resonance_mass(config_yml, params, resonance)
    width = resonance_width(config_yml, params, resonance)
    ma, mb = lineshape_mass_pair(config_yml, params, topology)
    q0_mass = bwr_ls_q0_mass(config_yml, params, resonance, topology, ma, mb)
    q0 = two_body_momentum_abs(q0_mass, ma, mb)
    channels = []
    for decay_ls in all_decay_ls:
        l = int(decay_ls[0])
        gamma = bwr_ls_gamma(params, resonance, l)
        ff0 = cascade_blatt_weisskopf_value(l, 3.0, q0_mass, ma, mb)
        gsq = m0 * m0 * width * gamma * gamma / (2.0 * q0 * ff0 * ff0)
        channels.append({"gsq": float(gsq), "ma": ma, "mb": mb, "l": l, "d": 3.0})
    return channels


def constant_lineshape_value(config_yml: dict[str, Any], resonance: str) -> Any:
    block = {}
    for alias in parameter_aliases(resonance):
        if alias in config_yml["particle"]:
            block = config_yml["particle"][alias]
            break
    return block.get("C", 1.0)


def make_builtin_or_custom_lineshape(
    config_yml: dict[str, Any],
    params: dict[str, Any],
    resonance: str,
    decay_ls: tuple[int, int],
    topology: list[Any],
    warnings_out: list[str],
    all_decay_ls: list[tuple[int, int]] | None = None,
) -> dict[str, Any]:
    info = particle_info(config_yml, params, resonance)
    model = info.get("model", "one")
    l = int(decay_ls[0])
    name = function_name_for_resonance(resonance) if "BWR_LS" in model else function_name_for_resonance(resonance, f"_l{l}")

    if "BWR_LS" in model:
        return {
            "name": name,
            "type": "MultichannelBreitWigner",
            "mass": resonance_mass(config_yml, params, resonance),
            "channels": multichannel_breit_wigner_channels(
                config_yml, params, resonance, topology, all_decay_ls or [decay_ls]
            ),
            "x": invariant_mass_squared_variable(topology),
        }

    if "BWR" in model:
        ma_name, mb_name = lineshape_mass_names(topology)
        return {
            "name": name,
            "type": "BreitWigner",
            "x": invariant_mass_squared_variable(topology),
            "mass": resonance_mass(config_yml, params, resonance),
            "width": resonance_width(config_yml, params, resonance),
            "ma": mass_value(config_yml, params, ma_name),
            "mb": mass_value(config_yml, params, mb_name),
            "l": l,
            "d": 3.0,
        }

    if "New" in model:
        warnings_out.append(f"{resonance}: TF-PWA nonresonant New model emitted as custom expression.")
        alpha = params.get(f"{resonance}_alpha", None)
        beta = params.get(f"{resonance}_beta", None)
        return {
            "name": name,
            "type": "custom",
            "expression": f"-exp(-(({alpha}) + i*({beta})) * (m^2 - m0^2))",
        }

    if "one" in model:
        return {"name": name, "type": "ConstantLineshape", "value": constant_lineshape_value(config_yml, resonance)}

    warnings_out.append(f"{resonance}: unsupported TF-PWA model {model!r}; emitted as custom placeholder.")
    return {"name": name, "type": "custom", "expression": f"unsupported TF-PWA model {model}"}

def unique_by_name(functions: Iterable[dict[str, Any]]) -> list[dict[str, Any]]:
    seen = {}
    for item in functions:
        seen.setdefault(item["name"], item)
    return [seen[name] for name in sorted(seen)]

In [ ]:
def root_ls_weight_key(resonance: str, topology: list[Any], root_idx: int) -> str:
    key_name = tfpwa_parameter_name(resonance)
    if topology == [[[1, 2], 3], 4]:
        return f"Bp->{key_name}.K_g_ls_{root_idx}"
    if topology == [[1, 2], [3, 4]]:
        return f"Bp->{key_name}.Dst_g_ls_{root_idx}"
    raise ValueError(f"Unsupported topology for root LS weight: {topology}")


def decay_ls_weight_key(resonance: str, topology: list[Any], decay_idx: int) -> str:
    key_name = tfpwa_parameter_name(resonance)
    if topology == [[[1, 2], 3], 4]:
        return f"{key_name}->Dst.D_g_ls_{decay_idx}"
    if topology == [[1, 2], [3, 4]]:
        return f"{key_name}->D.K_g_ls_{decay_idx}"
    raise ValueError(f"Unsupported topology for decay LS weight: {topology}")


def root_vertex_label(topology: list[Any]) -> str:
    if topology == [[[1, 2], 3], 4]:
        return "Bp_to_R_K"
    if topology == [[1, 2], [3, 4]]:
        return "Bp_to_R_Dst"
    raise ValueError(f"Unsupported topology for root vertex label: {topology}")


def decay_vertex_label(topology: list[Any]) -> str:
    if topology == [[[1, 2], 3], 4]:
        return "R_to_Dst_D"
    if topology == [[1, 2], [3, 4]]:
        return "R_to_D_K"
    raise ValueError(f"Unsupported topology for decay vertex label: {topology}")


def formfactor_for_step(
    step: dict[str, Any],
    resonance: str,
    vertex_label: str,
    ls: tuple[int, int],
    functions_out: list[dict[str, Any]],
) -> str:
    if not step.get("has_barrier_factor", False):
        return ""
    l = int(ls[0])
    name = blatt_weisskopf_function_name(resonance, vertex_label, l)
    functions_out.append(blatt_weisskopf_function(name, l, radius=3.0))
    return name


def root_convention_mismatch_factor(
    config_yml: dict[str, Any],
    params: dict[str, Any],
    resonance: str,
    topology: list[Any],
    root_ls: tuple[int, int],
    d: float = 3.0,
) -> float:
    m_bp = mass_value(config_yml, params, "Bp")
    m_k = mass_value(config_yml, params, "K")
    m_dst = mass_value(config_yml, params, "Dst")
    m_r = mass_value(config_yml, params, resonance)
    root_l = int(root_ls[0])
    if topology == [[[1, 2], 3], 4]:
        return convention_mismatch_factor_for_vertex(root_l, d, m_bp, m_r, m_k)
    if topology == [[1, 2], [3, 4]]:
        return convention_mismatch_factor_for_vertex(root_l, d, m_bp, m_r, m_dst)
    raise ValueError(f"Unsupported topology for root convention mismatch factor: {topology}")


def decay_convention_mismatch_factor(
    config_yml: dict[str, Any],
    params: dict[str, Any],
    resonance: str,
    topology: list[Any],
    decay_ls: tuple[int, int],
    d: float = 3.0,
) -> float:
    m_k = mass_value(config_yml, params, "K")
    m_d = mass_value(config_yml, params, "D")
    m_dst = mass_value(config_yml, params, "Dst")
    m_r = mass_value(config_yml, params, resonance)
    decay_l = int(decay_ls[0])
    if topology == [[[1, 2], 3], 4]:
        decay_m0 = decay_nominal_mass_for_mismatch(config_yml, params, resonance, m_dst, m_d)
        return convention_mismatch_factor_for_vertex(decay_l, d, decay_m0, m_dst, m_d)
    if topology == [[1, 2], [3, 4]]:
        return convention_mismatch_factor_for_vertex(decay_l, d, m_r, m_d, m_k)
    raise ValueError(f"Unsupported topology for decay convention mismatch factor: {topology}")


def weighted_ls_vertex(node: list[Any], ls: tuple[int, int], weight: complex, formfactor: str) -> dict[str, Any]:
    vertex = make_ls_vertex(node, ls, formfactor=formfactor)
    vertex["weight"] = complex_to_schema(weight)
    return vertex


def make_serialized_chain_components(
    config_yml: dict[str, Any],
    params: dict[str, Any],
    extracted_chain: dict[str, Any],
    warnings_out: list[str],
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    steps = extracted_chain["steps"]
    resonance = resonance_name_from_steps(steps)
    topology = chain_topology_from_steps(steps)
    resonance_node = resonance_node_for_topology(topology, resonance)
    root_node = root_node_for_topology(topology)
    root_step = steps[0]
    root_ls_list = root_step["ls_list"]
    resonance_decay_step = next(step for step in steps if step["core"] == resonance)
    decay_ls_list = resonance_decay_step["ls_list"]
    total_weight = param_complex_polar(params, total_weight_key(resonance, topology))

    functions = [make_builtin_or_custom_lineshape(config_yml, params, resonance, decay_ls, topology, warnings_out, decay_ls_list) for decay_ls in decay_ls_list]
    propagator_function = functions[0]["name"]

    root_vertices = []
    for root_idx, root_ls in enumerate(root_ls_list):
        root_weight = param_complex_polar(params, root_ls_weight_key(resonance, topology, root_idx), 1.0 + 0.0j)
        root_weight *= root_convention_mismatch_factor(config_yml, params, resonance, topology, root_ls)
        formfactor = formfactor_for_step(root_step, resonance, root_vertex_label(topology), root_ls, functions)
        root_vertices.append(weighted_ls_vertex(root_node, root_ls, root_weight, formfactor))

    decay_vertices = []
    for decay_idx, decay_ls in enumerate(decay_ls_list):
        decay_weight = param_complex_polar(params, decay_ls_weight_key(resonance, topology, decay_idx), 1.0 + 0.0j)
        if "BWR_LS" in particle_info(config_yml, params, resonance).get("model", "one"):
            decay_weight *= bwr_ls_gamma(params, resonance, int(decay_ls[0]))
        decay_weight *= decay_convention_mismatch_factor(config_yml, params, resonance, topology, decay_ls)
        formfactor = formfactor_for_step(resonance_decay_step, resonance, decay_vertex_label(topology), decay_ls, functions)
        decay_vertices.append(weighted_ls_vertex(resonance_node, decay_ls, decay_weight, formfactor))

    component = {
        "propagators": [
            {"spin": "1", "node": [1, 2], "parametrization": "constant_Dst"},
            {
                "spin": spin_for_particle(config_yml, params, resonance),
                "node": resonance_node,
                "parametrization": propagator_function,
            },
        ],
        "weight": complex_to_schema(total_weight),
        "vertices": [*root_vertices, *decay_vertices, make_dst_vertex()],
        "topology": topology,
        "name": resonance,
    }
    return [component], functions

def build_functions_and_chains(
    config_yml: dict[str, Any],
    params: dict[str, Any],
    chain_steps: list[dict[str, Any]],
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], list[str]]:
    warnings_out = []
    chains = []
    functions = [{"name": "constant_Dst", "type": "ConstantLineshape", "value": 1.0}]
    for extracted_chain in chain_steps:
        new_chains, new_functions = make_serialized_chain_components(config_yml, params, extracted_chain, warnings_out)
        chains.extend(new_chains)
        functions.extend(new_functions)
    chains = sorted(chains, key=lambda item: item["name"])
    return chains, unique_by_name(functions), warnings_out

In [ ]:
def build_parameter_point(params: dict[str, Any], name: str = "final_params_full") -> dict[str, Any]:
    return {
        "name": name,
        "parameters": [
            {"name": key, "value": float(value)}
            for key, value in sorted(params.items())
            if isinstance(value, (int, float))
        ],
    }


def build_domains(config_yml: dict[str, Any], final_state_order: list[str]) -> list[dict[str, Any]]:
    masses = tfpwa_nominal_final_masses(config_yml, list(config_yml["data"]["dat_order"]))
    m_parent = float(config_yml["particle"]["$top"]["Bp"]["mass"])
    return [
        {
            "name": "nominal_phase_space",
            "type": "product_domain",
            "parameters": [
                {"name": "m_parent", "value": m_parent},
                *[{"name": f"m_{name}", "value": float(masses[name])} for name in final_state_order],
            ],
            "axes": [
                {"name": "m2_D0pi", "min": (masses["D0"] + masses["pi"]) ** 2, "max": (m_parent - masses["D"] - masses["K"]) ** 2},
                {"name": "m2_DK", "min": (masses["D"] + masses["K"]) ** 2, "max": (m_parent - masses["D0"] - masses["pi"]) ** 2},
                {"name": "m2_D0piD", "min": (masses["D0"] + masses["pi"] + masses["D"]) ** 2, "max": (m_parent - masses["K"]) ** 2},
            ],
        }
    ]


def iter_topology_leaves(topology: Any) -> list[int]:
    if isinstance(topology, int):
        return [topology]
    leaves = []
    for item in topology:
        leaves.extend(iter_topology_leaves(item))
    return leaves


def deduplicate_preserve_order(items: Iterable[str]) -> list[str]:
    return list(dict.fromkeys(items))


def validate_references(model: dict[str, Any]) -> list[str]:
    issues = []
    required_top_level = {"distributions", "domains", "functions", "misc", "parameter_points"}
    missing = required_top_level - set(model)
    if missing:
        issues.append(f"Missing top-level keys: {sorted(missing)}")

    function_names = {f["name"] for f in model.get("functions", [])}
    for distribution in model.get("distributions", []):
        decay_description = distribution.get("decay_description", {})
        final_indices = {state["index"] for state in decay_description.get("kinematics", {}).get("final_state", [])}
        for chain in decay_description.get("chains", []):
            leaves = set(iter_topology_leaves(chain["topology"]))
            if leaves != final_indices:
                issues.append(f"Topology leaves {sorted(leaves)} do not match final-state indices {sorted(final_indices)} in {chain['name']}")
            for propagator in chain["propagators"]:
                if propagator["parametrization"] not in function_names:
                    issues.append(f"Missing function {propagator['parametrization']} used by {chain['name']}")
            for vertex in chain["vertices"]:
                vertex_leaves = set(iter_topology_leaves(vertex["node"]))
                if not vertex_leaves.issubset(final_indices):
                    issues.append(f"Vertex node {vertex['node']} in {chain['name']} references unknown leaves")
    return issues


def validate_schema_if_available(model: dict[str, Any], schema_path: Path | None = None) -> list[str]:
    if schema_path is None or not schema_path.exists():
        return ["Schema validation skipped: no local schema_path was provided."]
    try:
        import jsonschema
    except ImportError:
        return ["Schema validation skipped: jsonschema is not installed in this kernel."]
    schema = json.loads(schema_path.read_text(encoding="utf-8"))
    jsonschema.validate(model, schema)
    return ["Schema validation passed."]

In [ ]:
def convert_tfpwa_to_amplitude_serialization(
    paths: ConverterPaths,
    final_state_order: list[str],
    reference_tfpwa: dict[str, Any] | None = None,
) -> dict[str, Any]:
    config_yml = load_tfpwa_config_yaml(paths.config_path)
    params = load_params(paths.params_path)
    extracted_chains = extract_chain_steps(paths)
    chains, functions, warnings_out = build_functions_and_chains(config_yml, params, extracted_chains)
    warnings_out = deduplicate_preserve_order(warnings_out)

    model = {
        "distributions": [
            {
                "name": "Bplus_to_D0_pi_D_K_unpolarized_intensity",
                "type": "HadronicUnpolarizedIntensity",
                "decay_description": {
                    "kinematics": build_kinematics(config_yml, params, final_state_order),
                    "reference_topology": REFERENCE_TOPOLOGY,
                    "chains": chains,
                    "appendix": {
                        "source": "Converted from TF-PWA config_a.yml and final_params_full.json",
                        "tfpwa_dat_order": list(config_yml["data"]["dat_order"]),
                        "serialization_final_state_order": final_state_order,
                        "known_warnings": warnings_out,
                        "format_note": "Uses the official amplitude-serialization top-level schema with distributions/domains/functions/parameter_points.",
                        "weight_note": "Chain weight stores the TF-PWA total resonance coefficient. LS-vertex weights store the LS coupling coefficient times the nominal Blatt-Weisskopf convention mismatch factor for that vertex. The formfactor field stores the event-dependent Blatt-Weisskopf object inferred from TF-PWA barrier-factor metadata, or an empty string when no barrier factor is applied.",
                    },
                },
                "variables": [
                    {"node": [1, 2], "mass_phi_costheta": ["m_Dst", "phi_Dst", "cos_theta_Dst"]},
                    {"node": [[1, 2], 3], "mass_phi_costheta": ["m_DstD", "phi_DstD", "cos_theta_DstD"]},
                    {"node": [3, 4], "mass_phi_costheta": ["m_DK", "phi_DK", "cos_theta_DK"]},
                    {"node": [[1, 2], [3, 4]], "mass_phi_costheta": ["m_Dst_DK", "phi_Dst_DK", "cos_theta_Dst_DK"]},
                ],
                "parameters": [],
            }
        ],
        "domains": build_domains(config_yml, final_state_order),
        "functions": functions,
        "misc": {
            "amplitude_model_checksums": [],
            "reference_tfpwa": reference_tfpwa,
            "warnings": warnings_out,
        },
        "parameter_points": [build_parameter_point(params)],
    }

    issues = validate_references(model)
    if issues:
        raise ValueError("Reference validation failed:\n" + "\n".join(issues))
    return model


def write_serialization_json(model: dict[str, Any], output_path: Path) -> None:
    output_path.write_text(json.dumps(model, indent=2, sort_keys=False), encoding="utf-8")

## Build and export the JSON

The first call builds the JSON in memory. The explicit `write_serialization_json` call is the only file-writing step.

In [ ]:
serialization_model = convert_tfpwa_to_amplitude_serialization(
    PATHS,
    FINAL_STATE_ORDER,
    reference_tfpwa=reference_tfpwa,
)
write_serialization_json(serialization_model, PATHS.output_path)

decay_description = serialization_model["distributions"][0]["decay_description"]
print("Wrote:", PATHS.output_path)
print("Chains:", len(decay_description["chains"]))
print("Functions:", len(serialization_model["functions"]))
print("Parameter points:", len(serialization_model["parameter_points"]))
print("Schema check:")
for message in validate_schema_if_available(serialization_model):
    print(" -", message)
print("Warnings:")
for warning in serialization_model["misc"]["warnings"]:
    print(" -", warning)

## Inspect a few generated chains

In [ ]:
for chain in decay_description["chains"][:5]:
    print(json.dumps(chain, indent=2)[:1400])
    print("---")